In [ ]:
from __future__ import annotations
import json
import re
from pathlib import Path
import pandas as pd
import argparse
import pathlib
import os

import numpy as np

import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import seaborn as sns

import sys 

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from optimization import optimization_utils


def set_fontsize(base_fontsize=15):
    fontsize = base_fontsize
    plt.rcParams.update({
        'font.size': fontsize,
        'axes.titlesize': fontsize * 1,
        'axes.labelsize': fontsize,
        'xtick.labelsize': fontsize * 0.8,
        'ytick.labelsize': fontsize * 0.8,
        'legend.fontsize': fontsize * 0.8,
        'font.family': "Arial"
    })

plt.style.use('default')

set_fontsize()

In [ ]:
town = "Bonn"
objective = "cases_and_conc"
path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}"

In [ ]:
town = "Bonn"
objective = "cases_and_conc"
path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}"
cutoff_value = 0.05

phase_cut_dates = ["2024-01-31", "2024-01-03", "2023-11-08", "2023-07-19", "2023-03-15"]

df_metrics = pd.DataFrame()

phase_cut_date = phase_cut_dates[0]

for phase_cut_date in phase_cut_dates:
    df = pd.read_csv(f"{path}/multistart_results/{phase_cut_date}_{objective}/multistart_metrics_{phase_cut_date}.csv")
    df = df.loc[df.k1.notna()]
    train_val_negll_c = (df["train_negll_c"]*df["n_obs_train_c"] + df["val_negll_c"]*df["n_obs_val_c"])/(df["n_obs_train_c"] + df["n_obs_val_c"])
    train_val_negll_I = (df["train_negll_I"]*df["n_obs_train_I"] + df["val_negll_I"]*df["n_obs_val_I"])/(df["n_obs_train_I"] + df["n_obs_val_I"])
    df.loc[:,"train_val_negll"] = train_val_negll_c + train_val_negll_I
    df_sub = df.loc[(train_val_negll_c <= train_val_negll_c.quantile(0.25)) & (train_val_negll_I <= train_val_negll_I.quantile(0.25))].sort_values("train_val_negll")

    df_nsmallest = df_sub.nsmallest(int(len(df) * cutoff_value), 'train_val_negll')
    df_nsmallest = df_nsmallest[["total_negll_I", "total_negll_c", "test_negll_I"]]
    df_nsmallest["phase_cut_date"] = phase_cut_date
    df_metrics = pd.concat([df_metrics, df_nsmallest], axis=0)


In [ ]:
df_metrics = df_metrics.groupby("phase_cut_date").median().reset_index()
df_metrics["weeks"] = [50, 32, 16, 8, 4]

In [ ]:
town = "Bonn"
objective = "cases_and_conc"
cutoff_value = 0.05 # cutoff value for ensemble member selection (fraction of best models)

path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}/multistart_results"

phase_cut_date = "2023-03-15"
shedding_curve_data = np.load(f"{path}/{phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_shedding_curve.npz")


# load data and base model
base_config = {
        # data selection settings
        "data_kwargs": {
            "town": "Bonn",
            "sampling_area": "North_South",
            "project": "both", # one of ESI_CorA, AMELAG
            "max_precipitation_subsetting": None, # one of None, dry, light_rain
            "substance_normalization": "flow", # one of None, PMMoV, flow
            "gene_target": "N1", # one of N1, N2
            "log_scale": True, # this only considers WW measurements, not case counts
        },

        "E0": 862.857, 
        "I0": 1294.286,
        "R0": 162092.04, # 92% of pop, based on https://www.rki.de/DE/Themen/Infektionskrankheiten/Infektionskrankheiten-A-Z/C/COVID-19-Pandemie/AK-Studien/Ergebnisse.html
        "phase_cut_date": phase_cut_date, # date to split data into two phases
        "dt": 0.2,
        "underreporting_model": "monotone_increasing"
}
hparams_path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}/optuna_best_{phase_cut_date}_{objective}/hparams.json"
with open(hparams_path) as f:
        hparams = json.load(f)
base_config.update(hparams)
data = optimization_utils.two_phase_integrative_model_load_data(base_config)


In [ ]:

fig, ax = plt.subplots(figsize=(15.1, 2.8), ncols= 3)

sns.regplot(x=df_metrics.weeks, y=df_metrics.test_negll_I, ci=None, color = "#3d85c6ff", line_kws=dict(color="#8B0000"), ax=ax[0])
# regplot adds the regression line as the last line on the axis
ax[0].lines[-1].set_label("linear regression\ncurve")
# if you also want the scatter in the legend:
ax[0].collections[0].set_label("model metrics")

ax[0].legend(loc=4)

sns.regplot(x=df_metrics.total_negll_c, y=df_metrics.test_negll_I, ci=None, color = "#3d85c6ff", line_kws=dict(color="#8B0000"), ax=ax[1])
ax[0].set_xlabel('Evaluation horizon [weeks]')
ax[0].set_ylabel('NLL of cases\n(test data, per\nobservation)')
ax[1].set_ylabel(None)
ax[1].set_xlabel('NLL of concentration\n(overall data, per observation)')
# plt.tight_layout()
# plt.savefig('negLL_scatter.png', dpi=300, bbox_inches='tight')

# add shedding kernel curve
s = jnp.arange(0, base_config["T_max"]+base_config["dt"], base_config["dt"])

shedding_curve_quantiles = {q: jnp.quantile(shedding_curve_data["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}

shedding_median = shedding_curve_quantiles[0.5]

ax[2].plot(s, shedding_median, c="black", label="Median")

Rt_low  = shedding_curve_quantiles[0.25]
Rt_high = shedding_curve_quantiles[0.75]
ax[2].fill_between(s, Rt_low, Rt_high, color="grey", alpha=0.45, label="50% CI")

Rt_low  = shedding_curve_quantiles[0.05]
Rt_high = shedding_curve_quantiles[0.95]
ax[2].fill_between(s, Rt_low, Rt_high, color="grey", alpha=0.3, label="90% CI")

Rt_low  = shedding_curve_quantiles[0.025]
Rt_high = shedding_curve_quantiles[0.975]
ax[2].fill_between(s, Rt_low, Rt_high, color="grey", alpha=0.15, label="95% CI")
ax[2].set_xlabel("Time since becoming infected [d]")
ax[2].set_ylabel(r"Shedding kernel")
ax[2].legend()

ax[1].set_yticklabels([])

plt.tight_layout()
plt.savefig("Figure2_b_and_c.png", dpi=300)

In [ ]:
df_metrics